## Phase 5A — Feature Reproducibility Audit

This notebook verifies whether the production URL feature extractor can reproduce the feature values used in the PhiUSIIL dataset.

Goal:
Raw URL → independently computed features → compare with dataset feature values.

This helps prevent training-serving skew between model training and real-world inference.

In [7]:
import pandas as pd
import numpy as np

# Load cleaned PhiUSIIL dataset
df = pd.read_csv("../data/processed/PhiUSIIL_clean.csv")

print("Dataset shape:", df.shape)
print("URL column available:", "URL" in df.columns)
print("Total missing URLs:", df["URL"].isnull().sum())

df.head(3)

Dataset shape: (235370, 60)
URL column available: True
Total missing URLs: 0


,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,URLCharProb,...,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label,NoOfDots,NoOfSlashes,SuspiciousKeywordCount,HasHyphenInDomain,PathDepth
0,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,0.061933,...,28,119,0,124,1,2,2,0,0,0
1,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,0.050207,...,8,39,0,217,1,2,2,0,1,0
2,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,0.064129,...,7,42,2,5,1,3,2,0,0,0


In [8]:
# Phase 5A - Test 1: URLLength reproducibility

for i in range(5):
    url = df.loc[i, "URL"]
    dataset_value = df.loc[i, "URLLength"]
    calculated_value = len(url)

    print("URL:", url)
    print("Dataset URLLength:", dataset_value)
    print("Calculated len(URL):", calculated_value)
    print("Match:", dataset_value == calculated_value)
    print("-" * 60)

URL: https://www.southbankmosaics.com
Dataset URLLength: 31
Calculated len(URL): 32
Match: False
------------------------------------------------------------
URL: https://www.uni-mainz.de
Dataset URLLength: 23
Calculated len(URL): 24
Match: False
------------------------------------------------------------
URL: https://www.voicefmradio.co.uk
Dataset URLLength: 29
Calculated len(URL): 30
Match: False
------------------------------------------------------------
URL: https://www.sfnmjournal.com
Dataset URLLength: 26
Calculated len(URL): 27
Match: False
------------------------------------------------------------
URL: https://www.rewildingargentina.org
Dataset URLLength: 33
Calculated len(URL): 34
Match: False
------------------------------------------------------------


In [9]:
# Inspect exact URL strings for hidden/trailing characters

for i in range(5):
    url = df.loc[i, "URL"]

    print("repr(URL):", repr(url))
    print("Length:", len(url))
    print("Last character:", repr(url[-1]))
    print("Dataset URLLength:", df.loc[i, "URLLength"])
    print("-" * 60)

repr(URL): 'https://www.southbankmosaics.com'
Length: 32
Last character: 'm'
Dataset URLLength: 31
------------------------------------------------------------
repr(URL): 'https://www.uni-mainz.de'
Length: 24
Last character: 'e'
Dataset URLLength: 23
------------------------------------------------------------
repr(URL): 'https://www.voicefmradio.co.uk'
Length: 30
Last character: 'k'
Dataset URLLength: 29
------------------------------------------------------------
repr(URL): 'https://www.sfnmjournal.com'
Length: 27
Last character: 'm'
Dataset URLLength: 26
------------------------------------------------------------
repr(URL): 'https://www.rewildingargentina.org'
Length: 34
Last character: 'g'
Dataset URLLength: 33
------------------------------------------------------------


In [10]:
# Test URLLength across the entire cleaned dataset

calculated_lengths = df["URL"].str.len()

difference = calculated_lengths - df["URLLength"]

print("Total URLs:", len(df))
print("Exact matches:", (difference == 0).sum())
print("Off by +1:", (difference == 1).sum())
print("Other differences:", (~difference.isin([0, 1])).sum())

print("\nDifference distribution:")
print(difference.value_counts().sort_index().head(20))

Total URLs: 235370
Exact matches: 48428
Off by +1: 186940
Other differences: 2

Difference distribution:
0      48428
1     186940
4          1
35         1
Name: count, dtype: int64


In [11]:
# Compare URLs where URLLength matches vs where it differs by 1

audit = df[["URL", "URLLength"]].copy()
audit["PythonLength"] = audit["URL"].str.len()
audit["Difference"] = audit["PythonLength"] - audit["URLLength"]

print("=== EXACT MATCH EXAMPLES ===")
for _, row in audit[audit["Difference"] == 0].head(10).iterrows():
    print(repr(row["URL"]))
    print("Dataset:", row["URLLength"], "| Python:", row["PythonLength"])
    print()

print("\n=== OFF BY +1 EXAMPLES ===")
for _, row in audit[audit["Difference"] == 1].head(10).iterrows():
    print(repr(row["URL"]))
    print("Dataset:", row["URLLength"], "| Python:", row["PythonLength"])
    print()

=== EXACT MATCH EXAMPLES ===
'http://www.shprakserf.gq'
Dataset: 24 | Python: 24

'http://att-103731-107123.weeblysite.com/'
Dataset: 40 | Python: 40

'https://hidok4f8zl.firebaseapp.com/'
Dataset: 35 | Python: 35

'https://pontosapontamentolu.com/gclid/=/c/?gclid=ishecabh95cvbhb1eiwagy6m4vn7url3tlj5m_nu__lcyj4n06pqquydbh-56148buwwbmv7k1qejboci-isjw5_kiq'
Dataset: 140 | Python: 140

'https://sabadonoappmagazine.com/'
Dataset: 32 | Python: 32

'https://ggsexpole.web.app/'
Dataset: 26 | Python: 26

'https://s3.amazonaws.com/appforest_uf/f1678949673383x832048620362898600/index%20%284%29.html'
Dataset: 92 | Python: 92

'https://att-104164.weeblysite.com/'
Dataset: 34 | Python: 34

'http://tkmowpikuk.owl4fsrch.club/vnafvra97w/?q=3717065149&id=100'
Dataset: 64 | Python: 64

'https://a04854e8-0cca-4b74-80e6-aacb14a4a5f7.id.repl.co'
Dataset: 55 | Python: 55


=== OFF BY +1 EXAMPLES ===
'https://www.southbankmosaics.com'
Dataset: 31 | Python: 32

'https://www.uni-mainz.de'
Dataset: 23 | Python:

In [12]:
# Investigate relationship between trailing slash and URLLength difference

audit["EndsWithSlash"] = audit["URL"].str.endswith("/")

comparison = pd.crosstab(
    audit["Difference"],
    audit["EndsWithSlash"]
)

print(comparison)

EndsWithSlash   False  True 
Difference                  
0               25120  23308
1              173725  13215
4                   0      1
35                  1      0


In [13]:
# Compare protocol with URLLength difference

audit["Protocol"] = df["URL"].str.extract(r"^(https?)://", expand=False)

print(pd.crosstab(
    audit["Difference"],
    audit["Protocol"]
))

Protocol     http   https
Difference               
0           18089   30339
1           33546  153394
4               0       1
35              0       1


In [14]:
# Inspect unusual URLLength mismatches

weird = audit[~audit["Difference"].isin([0, 1])]

for _, row in weird.iterrows():
    print("URL:", repr(row["URL"]))
    print("Dataset URLLength:", row["URLLength"])
    print("Python Length:", row["PythonLength"])
    print("Difference:", row["Difference"])
    print("-" * 80)

URL: 'https://22017-5502.s3.webspace.re/privatkunden/Ã£Å“berblick/sicherheit/notfallhilfe/firmenkunden/geschÃ£Â¤ftskunden/kontakt/clients/'
Dataset URLLength: 128
Python Length: 132
Difference: 4
--------------------------------------------------------------------------------
URL: 'https://vinted.6545657.xyz/p5ci5kt9Ã¢â‚¬â€¹Ã¢â‚¬â€¹Ã¢â‚¬â€¹Ã¢â‚¬â€¹Ã¢â‚¬â€¹Ã¢â‚¬â€¹Ã¢â‚¬â€¹/tsidom/7'
Dataset URLLength: 65
Python Length: 100
Difference: 35
--------------------------------------------------------------------------------


In [15]:
# Investigate URLLength together with Domain information

check = df[[
    "URL",
    "Domain",
    "URLLength",
    "DomainLength"
]].copy()

check["PythonURLLength"] = check["URL"].str.len()
check["URLDifference"] = check["PythonURLLength"] - check["URLLength"]

print("=== EXACT MATCH ===")

print(
    check[check["URLDifference"] == 0]
    .head(10)
    .to_string(index=False)
)

print("\n=== OFF BY +1 ===")

print(
    check[check["URLDifference"] == 1]
    .head(10)
    .to_string(index=False)
)

=== EXACT MATCH ===
                                                                                                                                         URL                                          Domain  URLLength  DomainLength  PythonURLLength  URLDifference
                                                                                                                    http://www.shprakserf.gq                               www.shprakserf.gq         24            17               24              0
                                                                                                    http://att-103731-107123.weeblysite.com/                att-103731-107123.weeblysite.com         40            32               40              0
                                                                                                         https://hidok4f8zl.firebaseapp.com/                      hidok4f8zl.firebaseapp.com         35            26               35            

In [16]:
# Phase 5A - Test 2: DomainLength reproducibility

domain_audit = df[["Domain", "DomainLength"]].copy()

domain_audit["CalculatedDomainLength"] = domain_audit["Domain"].str.len()

domain_audit["Difference"] = (
    domain_audit["CalculatedDomainLength"]
    - domain_audit["DomainLength"]
)

print("Total rows:", len(domain_audit))
print("Exact matches:", (domain_audit["Difference"] == 0).sum())
print("Mismatches:", (domain_audit["Difference"] != 0).sum())

print("\nDifference distribution:")
print(domain_audit["Difference"].value_counts().sort_index().head(20))

Total rows: 235370
Exact matches: 235370
Mismatches: 0

Difference distribution:
Difference
0    235370
Name: count, dtype: int64


In [17]:
# Phase 5A - Test 3: IsDomainIP reproducibility

import ipaddress

def calculate_is_domain_ip(domain):
    try:
        ipaddress.ip_address(domain)
        return 1
    except ValueError:
        return 0


ip_audit = df[["Domain", "IsDomainIP"]].copy()

ip_audit["CalculatedIsDomainIP"] = (
    ip_audit["Domain"].apply(calculate_is_domain_ip)
)

ip_audit["Match"] = (
    ip_audit["IsDomainIP"] == ip_audit["CalculatedIsDomainIP"]
)

print("Total rows:", len(ip_audit))
print("Exact matches:", ip_audit["Match"].sum())
print("Mismatches:", (~ip_audit["Match"]).sum())

print("\nDataset IsDomainIP distribution:")
print(ip_audit["IsDomainIP"].value_counts())

print("\nCalculated distribution:")
print(ip_audit["CalculatedIsDomainIP"].value_counts())

Total rows: 235370
Exact matches: 235331
Mismatches: 39

Dataset IsDomainIP distribution:
IsDomainIP
0    234734
1       636
Name: count, dtype: int64

Calculated distribution:
CalculatedIsDomainIP
0    234773
1       597
Name: count, dtype: int64


In [18]:
# Inspect IsDomainIP mismatches

ip_mismatches = ip_audit[~ip_audit["Match"]]

print("Total mismatches:", len(ip_mismatches))

print(
    ip_mismatches[
        ["Domain", "IsDomainIP", "CalculatedIsDomainIP"]
    ].to_string(index=False)
)

Total mismatches: 39
                                 Domain  IsDomainIP  CalculatedIsDomainIP
     64.47.167.72.host.secureserver.net           1                     0
   244.33.109.208.host.secureserver.net           1                     0
   244.33.109.208.host.secureserver.net           1                     0
  38.49.145.34.bc.googleusercontent.com           1                     0
     166.95.74.97.host.secureserver.net           1                     0
   244.33.109.208.host.secureserver.net           1                     0
                   191.252.178.240:8087           1                     0
   244.33.109.208.host.secureserver.net           1                     0
                     47.103.128.80:8085           1                     0
   244.33.109.208.host.secureserver.net           1                     0
   244.33.109.208.host.secureserver.net           1                     0
   50.188.109.208.host.secureserver.net           1                     0
   104.129.205.92

In [19]:
# Test whether PhiUSIIL detects an IPv4 pattern anywhere in Domain

import re

ipv4_pattern = r"(?:\d{1,3}\.){3}\d{1,3}"

def calculate_is_domain_ip_pattern(domain):
    return int(bool(re.search(ipv4_pattern, str(domain))))


ip_pattern_audit = df[["Domain", "IsDomainIP"]].copy()

ip_pattern_audit["CalculatedIsDomainIP"] = (
    ip_pattern_audit["Domain"].apply(calculate_is_domain_ip_pattern)
)

ip_pattern_audit["Match"] = (
    ip_pattern_audit["IsDomainIP"]
    == ip_pattern_audit["CalculatedIsDomainIP"]
)

print("Total rows:", len(ip_pattern_audit))
print("Exact matches:", ip_pattern_audit["Match"].sum())
print("Mismatches:", (~ip_pattern_audit["Match"]).sum())

print("\nCalculated distribution:")
print(ip_pattern_audit["CalculatedIsDomainIP"].value_counts())

Total rows: 235370
Exact matches: 235369
Mismatches: 1

Calculated distribution:
CalculatedIsDomainIP
0    234733
1       637
Name: count, dtype: int64


In [20]:
# Inspect the final IsDomainIP mismatch

final_ip_mismatch = ip_pattern_audit[
    ~ip_pattern_audit["Match"]
]

print(
    final_ip_mismatch[
        ["Domain", "IsDomainIP", "CalculatedIsDomainIP"]
    ].to_string(index=False)
)

                                          Domain  IsDomainIP  CalculatedIsDomainIP
area.12.34.21.23.findomestic.fersanuniformes.com           0                     1


In [21]:
# Phase 5A - Test 4: TLD and TLDLength

tld_audit = df[
    ["Domain", "TLD", "TLDLength"]
].copy()

# Simple hypothesis:
# TLD = everything after the final dot in Domain
tld_audit["CalculatedTLD"] = (
    tld_audit["Domain"]
    .astype(str)
    .str.rsplit(".", n=1)
    .str[-1]
)

tld_audit["TLDMatch"] = (
    tld_audit["TLD"].astype(str)
    == tld_audit["CalculatedTLD"]
)

print("=== TLD ===")
print("Total rows:", len(tld_audit))
print("Exact matches:", tld_audit["TLDMatch"].sum())
print("Mismatches:", (~tld_audit["TLDMatch"]).sum())

# Now test TLDLength against the dataset's existing TLD
tld_audit["CalculatedTLDLength"] = (
    tld_audit["TLD"].astype(str).str.len()
)

tld_audit["TLDLengthMatch"] = (
    tld_audit["TLDLength"]
    == tld_audit["CalculatedTLDLength"]
)

print("\n=== TLDLength ===")
print("Exact matches:", tld_audit["TLDLengthMatch"].sum())
print("Mismatches:", (~tld_audit["TLDLengthMatch"]).sum())

=== TLD ===
Total rows: 235370
Exact matches: 235370
Mismatches: 0

=== TLDLength ===
Exact matches: 235370
Mismatches: 0


In [22]:
# Phase 5A - Test 5: NoOfSubDomain inspection

subdomain_audit = df[
    ["Domain", "TLD", "NoOfSubDomain"]
].copy()

print("=== SAMPLE DOMAINS ===")

print(
    subdomain_audit[
        ["Domain", "TLD", "NoOfSubDomain"]
    ]
    .head(30)
    .to_string(index=False)
)

=== SAMPLE DOMAINS ===
                       Domain  TLD  NoOfSubDomain
     www.southbankmosaics.com  com              1
             www.uni-mainz.de   de              1
       www.voicefmradio.co.uk   uk              2
          www.sfnmjournal.com  com              1
   www.rewildingargentina.org  org              1
      www.globalreporting.org  org              1
           www.saffronart.com  com              1
           www.nerdscandy.com  com              1
       www.hyderabadonline.in   in              1
                  www.aap.org  org              1
   www.religionenlibertad.com  com              1
             www.teramill.com  com              1
         www.socialpolicy.org  org              1
                www.aoh61.com  com              1
          www.bulgariaski.com  com              1
            www.brightika.com  com              1
                www.motley.ie   ie              1
               www.funzine.hu   hu              1
               www.dixxon.c

In [23]:
# Test NoOfSubDomain hypothesis across entire dataset

subdomain_audit["CalculatedNoOfSubDomain"] = (
    subdomain_audit["Domain"].astype(str).str.count(r"\.") - 1
)

subdomain_audit["Match"] = (
    subdomain_audit["NoOfSubDomain"]
    == subdomain_audit["CalculatedNoOfSubDomain"]
)

print("Total rows:", len(subdomain_audit))
print("Exact matches:", subdomain_audit["Match"].sum())
print("Mismatches:", (~subdomain_audit["Match"]).sum())

print("\nDataset distribution:")
print(subdomain_audit["NoOfSubDomain"].value_counts().sort_index())

print("\nCalculated distribution:")
print(subdomain_audit["CalculatedNoOfSubDomain"].value_counts().sort_index())

Total rows: 235370
Exact matches: 235370
Mismatches: 0

Dataset distribution:
NoOfSubDomain
0      13904
1     178142
2      36288
3       5105
4       1560
5        336
6         21
7          5
8          4
10         5
Name: count, dtype: int64

Calculated distribution:
CalculatedNoOfSubDomain
0      13904
1     178142
2      36288
3       5105
4       1560
5        336
6         21
7          5
8          4
10         5
Name: count, dtype: int64


In [24]:
# Phase 5A - Test 6: Inspect obfuscation features

obfuscation_audit = df[
    [
        "URL",
        "HasObfuscation",
        "NoOfObfuscatedChar",
        "ObfuscationRatio"
    ]
].copy()

print("=== HAS OBFUSCATION = 1 ===")

print(
    obfuscation_audit[
        obfuscation_audit["HasObfuscation"] == 1
    ]
    .head(20)
    .to_string(index=False)
)

print("\n=== HAS OBFUSCATION = 0 ===")

print(
    obfuscation_audit[
        obfuscation_audit["HasObfuscation"] == 0
    ]
    .head(10)
    .to_string(index=False)
)

=== HAS OBFUSCATION = 1 ===
                                                                                                                                                                                                                                                                                                                                          URL  HasObfuscation  NoOfObfuscatedChar  ObfuscationRatio
                                                                                                                                                                                                                                                 https://s3.amazonaws.com/appforest_uf/f1678949673383x832048620362898600/index%20%284%29.html               1                   9             0.098
                                                                                                                                                                                                    

In [25]:
# Test obfuscation hypothesis across entire dataset

import re

def count_obfuscated_chars(url):
    # Each percent-encoded sequence such as %20 contains 3 characters
    encoded_sequences = re.findall(r"%[0-9A-Fa-f]{2}", str(url))
    return len(encoded_sequences) * 3


obfuscation_test = df[
    [
        "URL",
        "URLLength",
        "HasObfuscation",
        "NoOfObfuscatedChar",
        "ObfuscationRatio"
    ]
].copy()

obfuscation_test["CalculatedNoOfObfuscatedChar"] = (
    obfuscation_test["URL"].apply(count_obfuscated_chars)
)

obfuscation_test["CalculatedHasObfuscation"] = (
    obfuscation_test["CalculatedNoOfObfuscatedChar"] > 0
).astype(int)

print("=== HasObfuscation ===")
print(
    "Matches:",
    (
        obfuscation_test["HasObfuscation"]
        == obfuscation_test["CalculatedHasObfuscation"]
    ).sum()
)
print(
    "Mismatches:",
    (
        obfuscation_test["HasObfuscation"]
        != obfuscation_test["CalculatedHasObfuscation"]
    ).sum()
)

print("\n=== NoOfObfuscatedChar ===")
print(
    "Matches:",
    (
        obfuscation_test["NoOfObfuscatedChar"]
        == obfuscation_test["CalculatedNoOfObfuscatedChar"]
    ).sum()
)
print(
    "Mismatches:",
    (
        obfuscation_test["NoOfObfuscatedChar"]
        != obfuscation_test["CalculatedNoOfObfuscatedChar"]
    ).sum()
)

=== HasObfuscation ===
Matches: 234970
Mismatches: 400

=== NoOfObfuscatedChar ===
Matches: 234841
Mismatches: 529


In [26]:
# Inspect obfuscation mismatches

has_obf_mismatch = obfuscation_test[
    obfuscation_test["HasObfuscation"]
    != obfuscation_test["CalculatedHasObfuscation"]
]

char_obf_mismatch = obfuscation_test[
    obfuscation_test["NoOfObfuscatedChar"]
    != obfuscation_test["CalculatedNoOfObfuscatedChar"]
]

print("=== HasObfuscation mismatch examples ===")

print(
    has_obf_mismatch[
        [
            "URL",
            "HasObfuscation",
            "CalculatedHasObfuscation",
            "NoOfObfuscatedChar",
            "CalculatedNoOfObfuscatedChar"
        ]
    ]
    .head(20)
    .to_string(index=False)
)

print("\n=== NoOfObfuscatedChar mismatch examples ===")

print(
    char_obf_mismatch[
        [
            "URL",
            "HasObfuscation",
            "NoOfObfuscatedChar",
            "CalculatedNoOfObfuscatedChar"
        ]
    ]
    .head(20)
    .to_string(index=False)
)

=== HasObfuscation mismatch examples ===
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        URL  HasObfuscation  CalculatedHasObfuscation  NoOfObfuscatedChar  CalculatedNoOfObfuscatedChar
                                                                                                                                                                                                                                                                                                                                                                                                       

In [27]:
# Phase 5A - Test 7: NoOfLettersInURL

letters_audit = df[
    ["URL", "NoOfLettersInURL"]
].copy()

letters_audit["CalculatedLetters"] = (
    letters_audit["URL"]
    .astype(str)
    .apply(lambda url: sum(char.isalpha() for char in url))
)

letters_audit["Match"] = (
    letters_audit["NoOfLettersInURL"]
    == letters_audit["CalculatedLetters"]
)

print("Total rows:", len(letters_audit))
print("Exact matches:", letters_audit["Match"].sum())
print("Mismatches:", (~letters_audit["Match"]).sum())

print("\nDifference distribution:")
print(
    (
        letters_audit["CalculatedLetters"]
        - letters_audit["NoOfLettersInURL"]
    ).value_counts().sort_index().head(20)
)

Total rows: 235370
Exact matches: 0
Mismatches: 235370

Difference distribution:
4       9769
5      43335
6       5137
7      10420
8      30312
9     135932
10        80
11        34
12        13
13        23
14       291
15         7
16         7
17         1
18         3
19         1
21         2
24         1
25         1
85         1
Name: count, dtype: int64


In [28]:
# Inspect NoOfLettersInURL examples

letters_check = df[
    ["URL", "Domain", "NoOfLettersInURL"]
].copy()

letters_check["WholeURLLetters"] = (
    letters_check["URL"]
    .astype(str)
    .apply(lambda x: sum(c.isalpha() for c in x))
)

print(
    letters_check.head(20).to_string(index=False)
)

                               URL                     Domain  NoOfLettersInURL  WholeURLLetters
  https://www.southbankmosaics.com   www.southbankmosaics.com                18               27
          https://www.uni-mainz.de           www.uni-mainz.de                 9               18
    https://www.voicefmradio.co.uk     www.voicefmradio.co.uk                15               24
       https://www.sfnmjournal.com        www.sfnmjournal.com                13               22
https://www.rewildingargentina.org www.rewildingargentina.org                20               29
   https://www.globalreporting.org    www.globalreporting.org                17               26
        https://www.saffronart.com         www.saffronart.com                12               21
        https://www.nerdscandy.com         www.nerdscandy.com                12               21
    https://www.hyderabadonline.in     www.hyderabadonline.in                16               25
               https://www.aap

In [29]:
from urllib.parse import urlparse

def count_letters(text):
    return sum(c.isalpha() for c in str(text))

component_audit = df[
    ["URL", "Domain", "NoOfLettersInURL"]
].copy()

component_audit["URLWithoutProtocol"] = (
    component_audit["URL"]
    .str.replace(r"^https?://", "", regex=True)
)

component_audit["LettersWithoutProtocol"] = (
    component_audit["URLWithoutProtocol"].apply(count_letters)
)

component_audit["DomainLetters"] = (
    component_audit["Domain"].apply(count_letters)
)

component_audit["LettersAfterDomain"] = component_audit.apply(
    lambda row: count_letters(
        row["URLWithoutProtocol"][len(str(row["Domain"])):]
    ),
    axis=1
)

print(
    component_audit[
        [
            "URL",
            "NoOfLettersInURL",
            "LettersWithoutProtocol",
            "DomainLetters",
            "LettersAfterDomain"
        ]
    ].head(20).to_string(index=False)
)

                               URL  NoOfLettersInURL  LettersWithoutProtocol  DomainLetters  LettersAfterDomain
  https://www.southbankmosaics.com                18                      22             22                   0
          https://www.uni-mainz.de                 9                      13             13                   0
    https://www.voicefmradio.co.uk                15                      19             19                   0
       https://www.sfnmjournal.com                13                      17             17                   0
https://www.rewildingargentina.org                20                      24             24                   0
   https://www.globalreporting.org                17                      21             21                   0
        https://www.saffronart.com                12                      16             16                   0
        https://www.nerdscandy.com                12                      16             16             

In [30]:
# Test the observed NoOfLettersInURL pattern across the full dataset

component_audit["CalculatedLettersHypothesis"] = (
    component_audit["LettersWithoutProtocol"] - 4
)

component_audit["Difference"] = (
    component_audit["CalculatedLettersHypothesis"]
    - component_audit["NoOfLettersInURL"]
)

print("Total rows:", len(component_audit))
print(
    "Exact matches:",
    (component_audit["Difference"] == 0).sum()
)
print(
    "Mismatches:",
    (component_audit["Difference"] != 0).sum()
)

print("\nDifference distribution:")
print(
    component_audit["Difference"]
    .value_counts()
    .sort_index()
    .head(30)
)

Total rows: 235370
Exact matches: 164822
Mismatches: 70548

Difference distribution:
Difference
-4      50891
-3       7350
-1      11821
 0     164822
 1         91
 2         39
 3         15
 4         24
 5         15
 6        285
 7          6
 8          2
 9          2
 10         2
 12         2
 16         2
 76         1
Name: count, dtype: int64


In [31]:
# Phase 5A - Test 8: NoOfDegitsInURL

digits_audit = df[
    ["URL", "NoOfDegitsInURL"]
].copy()

digits_audit["CalculatedDigits"] = (
    digits_audit["URL"]
    .astype(str)
    .apply(lambda url: sum(char.isdigit() for char in url))
)

digits_audit["Difference"] = (
    digits_audit["CalculatedDigits"]
    - digits_audit["NoOfDegitsInURL"]
)

print("Total rows:", len(digits_audit))
print("Exact matches:", (digits_audit["Difference"] == 0).sum())
print("Mismatches:", (digits_audit["Difference"] != 0).sum())

print("\nDifference distribution:")
print(
    digits_audit["Difference"]
    .value_counts()
    .sort_index()
    .head(20)
)

Total rows: 235370
Exact matches: 234160
Mismatches: 1210

Difference distribution:
Difference
0    234160
1      1209
7         1
Name: count, dtype: int64


In [32]:
# Inspect NoOfDegitsInURL mismatches

digit_mismatches = digits_audit[
    digits_audit["Difference"] != 0
].copy()

print("Total mismatches:", len(digit_mismatches))

print(
    digit_mismatches[
        ["URL", "NoOfDegitsInURL", "CalculatedDigits", "Difference"]
    ]
    .head(20)
    .to_string(index=False)
)

Total mismatches: 1210
                                                                                                                                                                                                                                                                  URL  NoOfDegitsInURL  CalculatedDigits  Difference
                                                                                                                                                                                                          http://vinted-pl-gj32d.lanru.top/authorize/1662975258698217               17                18           1
                                                                                                                                                                                                                                             https://tiny.one/ing9488                3                 4           1
                                                  

In [33]:
# Phase 5A - Test 9: NoOfEqualsInURL

equals_audit = df[
    ["URL", "NoOfEqualsInURL"]
].copy()

equals_audit["CalculatedEquals"] = (
    equals_audit["URL"].astype(str).str.count("=")
)

equals_audit["Difference"] = (
    equals_audit["CalculatedEquals"]
    - equals_audit["NoOfEqualsInURL"]
)

print("Total rows:", len(equals_audit))
print("Exact matches:", (equals_audit["Difference"] == 0).sum())
print("Mismatches:", (equals_audit["Difference"] != 0).sum())

print("\nDifference distribution:")
print(
    equals_audit["Difference"]
    .value_counts()
    .sort_index()
    .head(20)
)

Total rows: 235370
Exact matches: 235121
Mismatches: 249

Difference distribution:
Difference
0    235121
1       249
Name: count, dtype: int64


In [34]:
# Inspect NoOfEqualsInURL mismatches

equals_mismatches = equals_audit[
    equals_audit["Difference"] != 0
]

print("Total mismatches:", len(equals_mismatches))

print(
    equals_mismatches[
        ["URL", "NoOfEqualsInURL", "CalculatedEquals", "Difference"]
    ]
    .head(20)
    .to_string(index=False)
)

Total mismatches: 249
                                                                                                                                                                                                                  URL  NoOfEqualsInURL  CalculatedEquals  Difference
                                                                                                                    https://nowgamesentergo.com/gala/index.php?userid=&a=vluxe6wpwmhaqkvucany77j9xsqx8ajvtfuwsdhtkga=                2                 3           1
                                                                                                   https://acompanhando-meusgastos-online.com/luiza/home.php?userid=&uri=jkluamrfhfwos4cafnmeh7pa1pzx12wuktqwv4swvyw=                2                 3           1
                                                                                                                             https://arubahost.it.sell.20-57-10-180.cprapid.com/it/and/fatturazione

In [35]:
# Test whether PhiUSIIL excludes a final trailing "="

def calculate_equals(url):
    url = str(url)

    count = url.count("=")

    if url.endswith("="):
        count -= 1

    return count


equals_test = df[
    ["URL", "NoOfEqualsInURL"]
].copy()

equals_test["CalculatedEquals"] = (
    equals_test["URL"].apply(calculate_equals)
)

equals_test["Match"] = (
    equals_test["NoOfEqualsInURL"]
    == equals_test["CalculatedEquals"]
)

print("Total rows:", len(equals_test))
print("Exact matches:", equals_test["Match"].sum())
print("Mismatches:", (~equals_test["Match"]).sum())

Total rows: 235370
Exact matches: 235079
Mismatches: 291


In [36]:
# Phase 5A - Test 10: NoOfQMarkInURL

qmark_audit = df[
    ["URL", "NoOfQMarkInURL"]
].copy()

qmark_audit["CalculatedQMarks"] = (
    qmark_audit["URL"].astype(str).str.count(r"\?")
)

qmark_audit["Difference"] = (
    qmark_audit["CalculatedQMarks"]
    - qmark_audit["NoOfQMarkInURL"]
)

print("Total rows:", len(qmark_audit))
print("Exact matches:", (qmark_audit["Difference"] == 0).sum())
print("Mismatches:", (qmark_audit["Difference"] != 0).sum())

print("\nDifference distribution:")
print(
    qmark_audit["Difference"]
    .value_counts()
    .sort_index()
    .head(20)
)

Total rows: 235370
Exact matches: 235358
Mismatches: 12

Difference distribution:
Difference
0    235358
1        12
Name: count, dtype: int64


In [37]:
# Inspect all NoOfQMarkInURL mismatches

qmark_mismatches = qmark_audit[
    qmark_audit["Difference"] != 0
]

print("Total mismatches:", len(qmark_mismatches))

print(
    qmark_mismatches[
        ["URL", "NoOfQMarkInURL", "CalculatedQMarks", "Difference"]
    ].to_string(index=False)
)

Total mismatches: 12
                                                                                                 URL  NoOfQMarkInURL  CalculatedQMarks  Difference
                 https://bafybeifcnbmqe53hcqtw34275ym7oxa6lqssxpeaeetqrtbjxnizy5xppe.ipfs.w3s.link/?               0                 1           1
                                                           https://sg.now-carousell.com/cab/4725011?               0                 1           1
http://oneiehruwjnvwq.itsaol.com/auth/realms/access/a1b2c3/11cbfe764491da7e907ea6d3c5d07b70/login1/?               0                 1           1
                       http://appmylogin.dynamic-dns.net/it/a1b2c3/4b6a7be36616d8f291a9d5e175ce4233?               0                 1           1
                                                                             https://7s-change.com/?               0                 1           1
                                         https://lps.meltingice-advrtsng.com/xhij_7227_1_es_pe_mi

In [38]:
# Test whether PhiUSIIL ignores a trailing "?"

def calculate_qmarks(url):
    url = str(url)

    count = url.count("?")

    if url.endswith("?"):
        count -= 1

    return count


qmark_test = df[
    ["URL", "NoOfQMarkInURL"]
].copy()

qmark_test["CalculatedQMarks"] = (
    qmark_test["URL"].apply(calculate_qmarks)
)

qmark_test["Match"] = (
    qmark_test["NoOfQMarkInURL"]
    == qmark_test["CalculatedQMarks"]
)

print("Total rows:", len(qmark_test))
print("Exact matches:", qmark_test["Match"].sum())
print("Mismatches:", (~qmark_test["Match"]).sum())

Total rows: 235370
Exact matches: 235324
Mismatches: 46


In [39]:
# Phase 5A - Test 11: NoOfAmpersandInURL

amp_audit = df[
    ["URL", "NoOfAmpersandInURL"]
].copy()

amp_audit["CalculatedAmpersands"] = (
    amp_audit["URL"].astype(str).str.count("&")
)

amp_audit["Difference"] = (
    amp_audit["CalculatedAmpersands"]
    - amp_audit["NoOfAmpersandInURL"]
)

print("Total rows:", len(amp_audit))
print("Exact matches:", (amp_audit["Difference"] == 0).sum())
print("Mismatches:", (amp_audit["Difference"] != 0).sum())

print("\nDifference distribution:")
print(
    amp_audit["Difference"]
    .value_counts()
    .sort_index()
    .head(20)
)

Total rows: 235370
Exact matches: 232205
Mismatches: 3165

Difference distribution:
Difference
-149     3
-120     1
-97      2
-52      2
-50      1
-34      1
-26      1
-24      4
-23      1
-18      3
-17      1
-16      1
-15      2
-14      2
-13      4
-11     84
-10      1
-9       2
-8      18
-7      18
Name: count, dtype: int64


In [40]:
# Inspect NoOfAmpersandInURL mismatches

amp_mismatches = amp_audit[
    amp_audit["Difference"] != 0
]

print("Total mismatches:", len(amp_mismatches))

print(
    amp_mismatches[
        [
            "URL",
            "NoOfAmpersandInURL",
            "CalculatedAmpersands",
            "Difference"
        ]
    ]
    .head(20)
    .to_string(index=False)
)

Total mismatches: 3165
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     URL  NoOfAmpersandInURL  CalculatedAmpersands  Difference
                                                                                                                                                                                                                                                                                                                                                                                            https://s3.amazonaws.com/appforest_uf/f1678949673383x83204862036289860

In [41]:
# Phase 5A - Test 12: IsHTTPS

https_audit = df[
    ["URL", "IsHTTPS"]
].copy()

https_audit["CalculatedIsHTTPS"] = (
    https_audit["URL"]
    .astype(str)
    .str.lower()
    .str.startswith("https://")
    .astype(int)
)

https_audit["Match"] = (
    https_audit["IsHTTPS"]
    == https_audit["CalculatedIsHTTPS"]
)

print("Total rows:", len(https_audit))
print("Exact matches:", https_audit["Match"].sum())
print("Mismatches:", (~https_audit["Match"]).sum())

print("\nDataset distribution:")
print(https_audit["IsHTTPS"].value_counts())

print("\nCalculated distribution:")
print(https_audit["CalculatedIsHTTPS"].value_counts())

Total rows: 235370
Exact matches: 234877
Mismatches: 493

Dataset distribution:
IsHTTPS
1    184228
0     51142
Name: count, dtype: int64

Calculated distribution:
CalculatedIsHTTPS
1    183735
0     51635
Name: count, dtype: int64


In [42]:
# Inspect IsHTTPS mismatches

https_mismatches = https_audit[
    ~https_audit["Match"]
].copy()

print("Total mismatches:", len(https_mismatches))

print(
    https_mismatches[
        ["URL", "IsHTTPS", "CalculatedIsHTTPS"]
    ]
    .head(30)
    .to_string(index=False)
)

Total mismatches: 493
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        URL  IsHTTPS  CalculatedIsHTTPS
                                                                                                                                                                          

In [43]:
# Test PhiUSIIL IsHTTPS hypothesis:
# Does the URL contain the string "https" anywhere?

https_contains_test = df[
    ["URL", "IsHTTPS"]
].copy()

https_contains_test["CalculatedIsHTTPS"] = (
    https_contains_test["URL"]
    .astype(str)
    .str.lower()
    .str.contains("https", regex=False)
    .astype(int)
)

https_contains_test["Match"] = (
    https_contains_test["IsHTTPS"]
    == https_contains_test["CalculatedIsHTTPS"]
)

print("Total rows:", len(https_contains_test))
print("Exact matches:", https_contains_test["Match"].sum())
print("Mismatches:", (~https_contains_test["Match"]).sum())

print("\nDataset distribution:")
print(https_contains_test["IsHTTPS"].value_counts())

print("\nCalculated distribution:")
print(https_contains_test["CalculatedIsHTTPS"].value_counts())

Total rows: 235370
Exact matches: 235370
Mismatches: 0

Dataset distribution:
IsHTTPS
1    184228
0     51142
Name: count, dtype: int64

Calculated distribution:
CalculatedIsHTTPS
1    184228
0     51142
Name: count, dtype: int64


In [44]:
# Phase 5A - Test 13: NoOfDots

dots_audit = df[
    ["URL", "NoOfDots"]
].copy()

dots_audit["CalculatedDots"] = (
    dots_audit["URL"]
    .astype(str)
    .str.count(r"\.")
)

dots_audit["Difference"] = (
    dots_audit["CalculatedDots"]
    - dots_audit["NoOfDots"]
)

print("Total rows:", len(dots_audit))
print("Exact matches:", (dots_audit["Difference"] == 0).sum())
print("Mismatches:", (dots_audit["Difference"] != 0).sum())

print("\nDifference distribution:")
print(
    dots_audit["Difference"]
    .value_counts()
    .sort_index()
    .head(20)
)

Total rows: 235370
Exact matches: 235370
Mismatches: 0

Difference distribution:
Difference
0    235370
Name: count, dtype: int64


In [45]:
# Phase 5A - Test 14: NoOfSlashes

slashes_audit = df[
    ["URL", "NoOfSlashes"]
].copy()

slashes_audit["CalculatedSlashes"] = (
    slashes_audit["URL"]
    .astype(str)
    .str.count("/")
)

slashes_audit["Difference"] = (
    slashes_audit["CalculatedSlashes"]
    - slashes_audit["NoOfSlashes"]
)

print("Total rows:", len(slashes_audit))
print("Exact matches:", (slashes_audit["Difference"] == 0).sum())
print("Mismatches:", (slashes_audit["Difference"] != 0).sum())

print("\nDifference distribution:")
print(
    slashes_audit["Difference"]
    .value_counts()
    .sort_index()
    .head(20)
)

Total rows: 235370
Exact matches: 235370
Mismatches: 0

Difference distribution:
Difference
0    235370
Name: count, dtype: int64


In [46]:
# Phase 5A - Test 15: SuspiciousKeywordCount

SUSPICIOUS_KEYWORDS = [
    "login",
    "verify",
    "secure",
    "account",
    "update",
    "signin",
    "banking",
    "confirm",
    "password"
]

def calculate_suspicious_keyword_count(url):
    url = str(url).lower()

    return sum(
        keyword in url
        for keyword in SUSPICIOUS_KEYWORDS
    )


keyword_audit = df[
    ["URL", "SuspiciousKeywordCount"]
].copy()

keyword_audit["CalculatedKeywordCount"] = (
    keyword_audit["URL"].apply(
        calculate_suspicious_keyword_count
    )
)

keyword_audit["Match"] = (
    keyword_audit["SuspiciousKeywordCount"]
    == keyword_audit["CalculatedKeywordCount"]
)

print("Total rows:", len(keyword_audit))
print("Exact matches:", keyword_audit["Match"].sum())
print("Mismatches:", (~keyword_audit["Match"]).sum())

Total rows: 235370
Exact matches: 235370
Mismatches: 0


In [48]:
# Phase 5A - Test 16: HasHyphenInDomain

from urllib.parse import urlparse

def calculate_has_hyphen_in_domain(url):
    hostname = urlparse(str(url)).hostname

    if hostname is None:
        return 0

    return int("-" in hostname)


hyphen_audit = df[
    ["URL", "HasHyphenInDomain"]
].copy()

hyphen_audit["CalculatedHasHyphen"] = (
    hyphen_audit["URL"].apply(calculate_has_hyphen_in_domain)
)

hyphen_audit["Match"] = (
    hyphen_audit["HasHyphenInDomain"]
    == hyphen_audit["CalculatedHasHyphen"]
)

print("Total rows:", len(hyphen_audit))
print("Exact matches:", hyphen_audit["Match"].sum())
print("Mismatches:", (~hyphen_audit["Match"]).sum())

Total rows: 235370
Exact matches: 235370
Mismatches: 0


In [49]:
# Phase 5A - Test 17: PathDepth

from urllib.parse import urlparse

def calculate_path_depth(url):
    path = urlparse(str(url)).path
    parts = [part for part in path.split("/") if part]

    return len(parts)


path_audit = df[
    ["URL", "PathDepth"]
].copy()

path_audit["CalculatedPathDepth"] = (
    path_audit["URL"].apply(calculate_path_depth)
)

path_audit["Match"] = (
    path_audit["PathDepth"]
    == path_audit["CalculatedPathDepth"]
)

print("Total rows:", len(path_audit))
print("Exact matches:", path_audit["Match"].sum())
print("Mismatches:", (~path_audit["Match"]).sum())

Total rows: 235370
Exact matches: 235370
Mismatches: 0


## Phase 5A Audit Summary


### Feature Reproducibility Status

| Feature | Status | Finding |
|---|---|---|
| URLLength | Dataset-specific mismatch | Python `len(URL)` did not reproduce the dataset exactly. 48,428 rows matched, 186,940 differed by +1, and 2 encoding-corrupted URLs were larger outliers. Exact preprocessing used before PhiUSIIL feature generation could not be recovered reliably. |
| DomainLength | Verified | 100% reproducible |
| IsDomainIP | Verified* | Reproduced except for one anomalous dataset row |
| TLD | Verified | 100% reproducible |
| TLDLength | Verified | 100% reproducible |
| NoOfSubDomain | Verified | 100% reproducible |
| HasObfuscation | Near-exact / dataset-specific mismatch | Percent-encoding based reproduction matched 234,970 of 235,370 rows, but 400 rows remained inconsistent with the stored dataset values. |
| NoOfObfuscatedChar | Near-exact / dataset-specific mismatch | Percent-encoding based reproduction matched 234,841 of 235,370 rows, but 529 rows remained inconsistent. Exact original preprocessing could not be established safely. |
| ObfuscationRatio | Unresolved | Depends on NoOfObfuscatedChar and the dataset's URL-length/preprocessing definition; therefore exact production reproduction is not yet guaranteed. |
| NoOfLettersInURL | Dataset-specific mismatch | Direct alphabetic-character counting from the stored raw URL did not reproduce the PhiUSIIL column reliably, indicating a different URL representation or preprocessing during original feature generation. |
| LetterRatioInURL | Unresolved | Depends on the unresolved NoOfLettersInURL and URL-length/preprocessing definitions, so exact production reproduction cannot currently be guaranteed. |
| NoOfDegitsInURL | Near-exact / dataset-specific mismatch | Direct digit counting matched 234,160 of 235,370 rows; 1,210 rows remained inconsistent, with most mismatches differing by only one digit. |
| DegitRatioInURL | Unresolved | Depends on NoOfDegitsInURL and the dataset's URL-length/preprocessing definition, so exact reproduction cannot currently be guaranteed. |
| NoOfEqualsInURL | Near-exact / dataset-specific mismatch | Direct `=` counting matched 235,121 of 235,370 rows; 249 rows remained inconsistent with the stored dataset values. |
| NoOfQMarkInURL | Near-exact / dataset-specific mismatch | Direct `?` counting matched 235,358 of 235,370 rows; only 12 rows remained inconsistent. |
| NoOfAmpersandInURL | Dataset-specific mismatch | Direct `&` counting matched 232,205 of 235,370 rows, with 3,165 mismatches and large differences on some URLs, indicating preprocessing or representation differences. |
| NoOfOtherSpecialCharsInURL | Unresolved / dataset-specific mismatch | Multiple plausible raw-URL definitions were tested, but none reproduced the PhiUSIIL column reliably. The exact original special-character definition/preprocessing could not be established safely. |
| SpacialCharRatioInURL | Unresolved | Depends on NoOfOtherSpecialCharsInURL and the dataset's URL-length/preprocessing logic; therefore exact production reproduction cannot currently be guaranteed. |
| IsHTTPS |  Verified | 100% reproduced using `"https" in url.lower()` |
| NoOfDots |  Verified | 100% reproduced using `URL.count(".")` |
| NoOfSlashes |  Verified | 100% reproduced using `URL.count("/")` |
| SuspiciousKeywordCount | Verified | 100% reproduced using original Phase 3 keyword logic |
| HasHyphenInDomain | Verified | 100% reproduced using original Phase 3 hostname logic |
| PathDepth | Verified | 100% reproduced using original Phase 3 path logic |

### Important Conclusion

Phase 5A revealed that several PhiUSIIL-native lexical features cannot safely be reproduced by assuming their column names directly describe their implementation.

Therefore, TrustLens will not blindly implement guessed formulas.

Features with exact reproducibility can be safely transferred to the production feature extractor. Ambiguous features require a final production-feature decision before the extractor and saved model are finalized.

This audit helps prevent training-serving skew between the dataset used during model development and URLs processed by the deployed TrustLens application.
# Phase 5A - Test 18:
# Inspect NoOfOtherSpecialCharsInURL

special_audit = df[
    [
        "URL",
        "NoOfOtherSpecialCharsInURL",
        "SpacialCharRatioInURL"
    ]
].copy()

print("=== LOW SPECIAL-CHARACTER COUNTS ===")

print(
    special_audit
    .sort_values("NoOfOtherSpecialCharsInURL")
    .head(15)
    .to_string(index=False)
)

print("\n=== HIGH SPECIAL-CHARACTER COUNTS ===")

print(
    special_audit
    .sort_values(
        "NoOfOtherSpecialCharsInURL",
        ascending=False
    )
    .head(15)
    .to_string(index=False)
)

In [51]:
# Phase 5A - Test 19:
# Test NoOfOtherSpecialCharsInURL

import re

def calculate_other_special_chars(url):
    url = str(url)

    # Remove letters, digits and common URL structural characters
    cleaned = re.sub(r"[A-Za-z0-9./:?=&_-]", "", url)

    return len(cleaned)


special_test = df[
    ["URL", "NoOfOtherSpecialCharsInURL"]
].copy()

special_test["CalculatedSpecialChars"] = (
    special_test["URL"].apply(calculate_other_special_chars)
)

special_test["Difference"] = (
    special_test["CalculatedSpecialChars"]
    - special_test["NoOfOtherSpecialCharsInURL"]
)

print("Total rows:", len(special_test))

print(
    "Exact matches:",
    (special_test["Difference"] == 0).sum()
)

print(
    "Mismatches:",
    (special_test["Difference"] != 0).sum()
)

print("\nDifference distribution:")
print(
    special_test["Difference"]
    .value_counts()
    .sort_index()
    .head(30)
)

Total rows: 235370
Exact matches: 13
Mismatches: 235357

Difference distribution:
Difference
-402     1
-373     1
-296     1
-266     2
-105     1
-65      1
-63      1
-61      2
-60      1
-59      3
-58      5
-57      2
-55      6
-54      2
-52      1
-51      1
-50      1
-48      1
-47      1
-46      1
-45      1
-44      3
-43      1
-41      6
-40      4
-39      2
-38      2
-37      6
-36      5
-35     10
Name: count, dtype: int64


In [52]:
# Phase 5A - Diagnostic:
# Discover characters used by NoOfOtherSpecialCharsInURL

from collections import Counter

# Count every non-alphanumeric character appearing in all URLs
char_counter = Counter()

for url in df["URL"].astype(str):
    for ch in url:
        if not ch.isalnum():
            char_counter[ch] += 1

print("Non-alphanumeric characters found in dataset:\n")

for char, count in char_counter.most_common():
    print(repr(char), "->", count)

Non-alphanumeric characters found in dataset:

'/' -> 572918
'.' -> 531229
':' -> 236530
'-' -> 82155
'=' -> 14919
'&' -> 8514
'_' -> 8221
'?' -> 6937
'%' -> 5903
';' -> 5218
'@' -> 1726
'#' -> 869
'+' -> 315
',' -> 139
'*' -> 138
'(' -> 108
')' -> 99
'~' -> 73
'!' -> 48
'$' -> 35
']' -> 21
'[' -> 20
'¢' -> 7
'‚' -> 7
'¬' -> 7
'€' -> 7
"'" -> 4
'£' -> 2
'“' -> 1
'¤' -> 1


In [53]:
# Phase 5A - Diagnostic:
# Compare individual special-character counts with
# NoOfOtherSpecialCharsInURL

candidate_chars = [
    "/", ".", ":", "-", "=", "&", "_", "?", "%",
    ";", "@", "#", "+", ",", "*", "(", ")", "~",
    "!", "$", "]", "[", "'", "£", "¢", "‚", "¬",
    "€", "“", "¤"
]

results = []

for ch in candidate_chars:

    calculated = df["URL"].astype(str).str.count(
        re.escape(ch)
    )

    exact_matches = (
        calculated == df["NoOfOtherSpecialCharsInURL"]
    ).sum()

    results.append({
        "Character": repr(ch),
        "ExactMatches": exact_matches,
        "MatchPercent": round(
            exact_matches / len(df) * 100, 4
        )
    })

char_results = pd.DataFrame(results)

print(
    char_results
    .sort_values("ExactMatches", ascending=False)
    .to_string(index=False)
)

Character  ExactMatches  MatchPercent
      ':'        130999       55.6566
      '/'         53024       22.5279
      '.'         21663        9.2038
      '%'            67        0.0285
      '='            19        0.0081
      '?'             6        0.0025
      '&'             4        0.0017
      '-'             4        0.0017
      '_'             4        0.0017
      ';'             4        0.0017
      '@'             4        0.0017
      '#'             4        0.0017
      '+'             4        0.0017
      ','             4        0.0017
      '*'             4        0.0017
      '('             4        0.0017
      ')'             4        0.0017
      '~'             4        0.0017
      '!'             4        0.0017
      '$'             4        0.0017
      ']'             4        0.0017
      '['             4        0.0017
      "'"             4        0.0017
      '£'             4        0.0017
      '¢'             4        0.0017
      '‚'   

In [54]:
# Phase 5A - Test 20:
# Test likely PhiUSIIL definition of NoOfOtherSpecialCharsInURL

def calculate_other_special_chars_v2(url):
    url = str(url)

    count = 0

    for ch in url:
        # Count special characters except the three
        # that PhiUSIIL stores in separate columns
        if not ch.isalnum() and ch not in ["=", "?", "&"]:
            count += 1

    return count


special_test_v2 = df[
    ["URL", "NoOfOtherSpecialCharsInURL"]
].copy()

special_test_v2["CalculatedSpecialChars"] = (
    special_test_v2["URL"].apply(
        calculate_other_special_chars_v2
    )
)

special_test_v2["Difference"] = (
    special_test_v2["CalculatedSpecialChars"]
    - special_test_v2["NoOfOtherSpecialCharsInURL"]
)

print("Total rows:", len(special_test_v2))

print(
    "Exact matches:",
    (special_test_v2["Difference"] == 0).sum()
)

print(
    "Mismatches:",
    (special_test_v2["Difference"] != 0).sum()
)

print("\nDifference distribution:")

print(
    special_test_v2["Difference"]
    .value_counts()
    .sort_index()
    .head(30)
)

Total rows: 235370
Exact matches: 267
Mismatches: 235103

Difference distribution:
Difference
-47         1
-26         9
-12         2
-10         1
-9          4
-8          6
-7          8
-6          2
-5          9
-4          8
-3         10
-2         31
-1        199
 0        267
 1        550
 2       1053
 3      43317
 4     188523
 5        826
 6        170
 7         59
 8         34
 9        121
 10        21
 11        17
 12         4
 13         2
 14         4
 15        81
 16         3
Name: count, dtype: int64


## Phase 5A Final Production Feature Decision

The reproducibility audit is complete for all 24 selected features.

The audit identified three categories:

1. **Exactly reproducible features** — their calculation can be transferred safely to production code.

2. **Near-exact features with limited dataset inconsistencies** — their intended calculation is clear, but a small number of stored PhiUSIIL rows differ.

3. **Unresolved dataset-specific features** — their exact original preprocessing or calculation could not be reproduced reliably from the stored raw URL.

No production feature will be removed, redefined, or replaced solely because of an unexplained dataset mismatch.

Before the production feature schema is frozen, the effect of the unresolved features on the trained Random Forest must be reviewed. The Phase 4/4.5 experimental model and notebook remain unchanged.

### Final Decision

Based on the full reproducibility audit, the production version of TrustLens will use a 17-feature URL feature set.

#### Keep — Exactly Reproducible

- DomainLength
- TLD
- TLDLength
- NoOfSubDomain
- IsHTTPS
- NoOfDots
- NoOfSlashes
- SuspiciousKeywordCount
- HasHyphenInDomain
- PathDepth

#### Keep — Clean Production Definition + Retrain

- IsDomainIP
- HasObfuscation
- NoOfObfuscatedChar
- NoOfDegitsInURL
- NoOfEqualsInURL
- NoOfQMarkInURL

These features showed very high agreement with the PhiUSIIL dataset. Instead of reproducing isolated dataset anomalies, TrustLens will use clear and deterministic definitions and regenerate these features consistently during both training and inference.

#### Redefine + Retrain

- URLLength

TrustLens will define URLLength as the Python length of the raw URL string:

`len(url)`

The production model will be retrained using this definition so that training and inference use identical logic.

#### Excluded from Production Model

- ObfuscationRatio
- NoOfLettersInURL
- LetterRatioInURL
- DegitRatioInURL
- NoOfAmpersandInURL
- NoOfOtherSpecialCharsInURL
- SpacialCharRatioInURL

These features could not be reproduced reliably from the stored raw URL using a clearly defensible definition. They are therefore excluded from the production model to prevent training-serving skew.

The original Phase 4/4.5 24-feature Random Forest remains unchanged as the experimental baseline.

The production model will later be retrained using features generated entirely by the TrustLens production feature extractor.

## Phase 5C — Production Extractor Validation

In [55]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.src.feature_extractor import extract_url_features

print("Feature extractor imported successfully.")

Feature extractor imported successfully.


In [56]:
# Phase 5C - Step 2:
# Validate production extractor on sample PhiUSIIL rows

production_features = [
    "URLLength",
    "DomainLength",
    "IsDomainIP",
    "TLD",
    "TLDLength",
    "NoOfSubDomain",
    "HasObfuscation",
    "NoOfObfuscatedChar",
    "NoOfDegitsInURL",
    "NoOfEqualsInURL",
    "NoOfQMarkInURL",
    "IsHTTPS",
    "NoOfDots",
    "NoOfSlashes",
    "SuspiciousKeywordCount",
    "HasHyphenInDomain",
    "PathDepth",
]

for i in range(5):
    url = df.loc[i, "URL"]
    extracted = extract_url_features(url)

    print(f"\n=== ROW {i} ===")
    print("URL:", url)

    for feature in production_features:
        dataset_value = df.loc[i, feature]
        extracted_value = extracted[feature]

        print(
            f"{feature}: "
            f"dataset={dataset_value} | "
            f"extracted={extracted_value}"
        )


=== ROW 0 ===
URL: https://www.southbankmosaics.com
URLLength: dataset=31 | extracted=32
DomainLength: dataset=24 | extracted=24
IsDomainIP: dataset=0 | extracted=0
TLD: dataset=com | extracted=com
TLDLength: dataset=3 | extracted=3
NoOfSubDomain: dataset=1 | extracted=1
HasObfuscation: dataset=0 | extracted=0
NoOfObfuscatedChar: dataset=0 | extracted=0
NoOfDegitsInURL: dataset=0 | extracted=0
NoOfEqualsInURL: dataset=0 | extracted=0
NoOfQMarkInURL: dataset=0 | extracted=0
IsHTTPS: dataset=1 | extracted=1
NoOfDots: dataset=2 | extracted=2
NoOfSlashes: dataset=2 | extracted=2
SuspiciousKeywordCount: dataset=0 | extracted=0
HasHyphenInDomain: dataset=0 | extracted=0
PathDepth: dataset=0 | extracted=0

=== ROW 1 ===
URL: https://www.uni-mainz.de
URLLength: dataset=23 | extracted=24
DomainLength: dataset=16 | extracted=16
IsDomainIP: dataset=0 | extracted=0
TLD: dataset=de | extracted=de
TLDLength: dataset=2 | extracted=2
NoOfSubDomain: dataset=1 | extracted=1
HasObfuscation: dataset=0 | 

In [58]:
# Phase 5C - Step 3:
# Run production extractor across the full cleaned dataset

extracted_df = df["URL"].apply(
    extract_url_features
).apply(pd.Series)

print("Extracted shape:", extracted_df.shape)

print("\nColumns:")
print(extracted_df.columns.tolist())

print("\nMissing values:")
print(extracted_df.isnull().sum())

extracted_df.head()

Extracted shape: (235370, 17)

Columns:
['URLLength', 'DomainLength', 'IsDomainIP', 'TLD', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'NoOfDegitsInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'IsHTTPS', 'NoOfDots', 'NoOfSlashes', 'SuspiciousKeywordCount', 'HasHyphenInDomain', 'PathDepth']

Missing values:
URLLength                 0
DomainLength              0
IsDomainIP                0
TLD                       0
TLDLength                 0
NoOfSubDomain             0
HasObfuscation            0
NoOfObfuscatedChar        0
NoOfDegitsInURL           0
NoOfEqualsInURL           0
NoOfQMarkInURL            0
IsHTTPS                   0
NoOfDots                  0
NoOfSlashes               0
SuspiciousKeywordCount    0
HasHyphenInDomain         0
PathDepth                 0
dtype: int64


,URLLength,DomainLength,IsDomainIP,TLD,TLDLength,NoOfSubDomain,HasObfuscation,NoOfObfuscatedChar,NoOfDegitsInURL,NoOfEqualsInURL,NoOfQMarkInURL,IsHTTPS,NoOfDots,NoOfSlashes,SuspiciousKeywordCount,HasHyphenInDomain,PathDepth
0,32,24,0,com,3,1,0,0,0,0,0,1,2,2,0,0,0
1,24,16,0,de,2,1,0,0,0,0,0,1,2,2,0,1,0
2,30,22,0,uk,2,2,0,0,0,0,0,1,3,2,0,0,0
3,27,19,0,com,3,1,0,0,0,0,0,1,2,2,0,0,0
4,34,26,0,org,3,1,0,0,0,0,0,1,2,2,0,0,0


In [59]:
# Phase 5C - Step 4:
# Compare production extractor values with original dataset values

comparison_results = []

for feature in production_features:
    if feature == "TLD":
        matches = (
            df[feature].astype(str)
            == extracted_df[feature].astype(str)
        )
    else:
        matches = (
            df[feature]
            == extracted_df[feature]
        )

    comparison_results.append({
        "Feature": feature,
        "Matches": matches.sum(),
        "Mismatches": (~matches).sum(),
        "MatchPercent": round(matches.mean() * 100, 4)
    })

comparison_summary = pd.DataFrame(comparison_results)

comparison_summary

,Feature,Matches,Mismatches,MatchPercent
0,URLLength,48428,186942,20.5753
1,DomainLength,235334,36,99.9847
2,IsDomainIP,235369,1,99.9996
3,TLD,235344,26,99.9890
4,TLDLength,235344,26,99.9890
5,NoOfSubDomain,235362,8,99.9966
6,HasObfuscation,234970,400,99.8301
7,NoOfObfuscatedChar,234841,529,99.7752
8,NoOfDegitsInURL,234160,1210,99.4859
9,NoOfEqualsInURL,235121,249,99.8942


In [60]:
# Phase 5C - Step 5:
# Build production-ready training dataset

production_df = extracted_df.copy()

# Keep the original raw URL for traceability
production_df.insert(0, "URL", df["URL"].values)

# Add target label
production_df["label"] = df["label"].values

print("Production dataset shape:", production_df.shape)
print("Missing values:", production_df.isnull().sum().sum())

print("\nColumns:")
print(production_df.columns.tolist())

production_df.head()

Production dataset shape: (235370, 19)
Missing values: 0

Columns:
['URL', 'URLLength', 'DomainLength', 'IsDomainIP', 'TLD', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'NoOfDegitsInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'IsHTTPS', 'NoOfDots', 'NoOfSlashes', 'SuspiciousKeywordCount', 'HasHyphenInDomain', 'PathDepth', 'label']


,URL,URLLength,DomainLength,IsDomainIP,TLD,TLDLength,NoOfSubDomain,HasObfuscation,NoOfObfuscatedChar,NoOfDegitsInURL,NoOfEqualsInURL,NoOfQMarkInURL,IsHTTPS,NoOfDots,NoOfSlashes,SuspiciousKeywordCount,HasHyphenInDomain,PathDepth,label
0,https://www.southbankmosaics.com,32,24,0,com,3,1,0,0,0,0,0,1,2,2,0,0,0,1
1,https://www.uni-mainz.de,24,16,0,de,2,1,0,0,0,0,0,1,2,2,0,1,0,1
2,https://www.voicefmradio.co.uk,30,22,0,uk,2,2,0,0,0,0,0,1,3,2,0,0,0,1
3,https://www.sfnmjournal.com,27,19,0,com,3,1,0,0,0,0,0,1,2,2,0,0,0,1
4,https://www.rewildingargentina.org,34,26,0,org,3,1,0,0,0,0,0,1,2,2,0,0,0,1


In [61]:
# Phase 5C - Step 6:
# Save production-ready dataset separately

output_path = "../data/processed/PhiUSIIL_production.csv"

production_df.to_csv(
    output_path,
    index=False
)

print("Production dataset saved successfully.")
print("Saved to:", output_path)

Production dataset saved successfully.
Saved to: ../data/processed/PhiUSIIL_production.csv


In [62]:
# Phase 5C - Step 7:
# Reload saved production dataset and verify

production_check = pd.read_csv(
    "../data/processed/PhiUSIIL_production.csv"
)

print("Reloaded shape:", production_check.shape)
print("Total missing values:", production_check.isnull().sum().sum())

print("\nColumns:")
print(production_check.columns.tolist())

print("\nLabel distribution:")
print(production_check["label"].value_counts())

Reloaded shape: (235370, 19)
Total missing values: 0

Columns:
['URL', 'URLLength', 'DomainLength', 'IsDomainIP', 'TLD', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'NoOfDegitsInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'IsHTTPS', 'NoOfDots', 'NoOfSlashes', 'SuspiciousKeywordCount', 'HasHyphenInDomain', 'PathDepth', 'label']

Label distribution:
label
1    134850
0    100520
Name: count, dtype: int64
